In [ ]:
using Plots, DifferentialEquations, LaTeXStrings
include("imode_sigmoid.jl")
using .ImodeSigmoid

In [ ]:
function SpikingModel!(du,u,p,t)

	Iapp, Ithr, Igain, Ilin = p

	sig_pars = (Ithr=Ithr, Igain=Igain, Ilin=Ilin)
	
	If, Is = u

	du[1] = (-If - Is + Imode_sigmoid_eval(If, sig_pars) + Iapp) / τ_f
	du[2] = (-Is + If) / τ_s

end

In [ ]:
function BurstingModel!(du,u,p,t)

	Iapp, Ithr_f, Igain_f, Ilin_f, Ithr_s, Igain_s, Ilin_s = p

	sigf_pars = (Ithr=Ithr_f, Igain=Igain_f, Ilin=Ilin_f)
	sigs_pars = (Ithr=Ithr_s, Igain=Igain_s, Ilin=Ilin_s)
	
	If, Is, Ius = u

	du[1] = (-If - Is - Ius + Imode_sigmoid_eval(If, sigf_pars) + Imode_sigmoid_eval(Is, sigs_pars) + Iapp) / τ_f
	du[2] = (-Is + If) / τ_s
	du[3] = (-Ius + If) / τ_us

	if If <= 0 && du[1] < 0
		du[1] = 0
	end

end

In [ ]:
function BurstingModel_inact!(du,u,p,t)

	Iapp, Ithr_f, Igain_f, Ilin_f, Ithr_s, Igain_s, Ilin_s = p
	
	If, Is, Ius = u

	sigf_pars = (Ithr=Ithr_f, Igain=Igain_f-Is, Ilin=Ilin_f)
	sigs_pars = (Ithr=Ithr_s, Igain=Igain_s-Ius, Ilin=Ilin_s)

	du[1] = (-If - Is - Ius + Imode_sigmoid_eval(If, sigf_pars, var_gain=true) + Imode_sigmoid_eval(Is, sigs_pars, var_gain=true) + Iapp) / τ_f
	du[2] = (-Is + If) / τ_s
	du[3] = (-Ius + If) / τ_us

	if If <= 0 && du[1] < 0
		du[1] = 0
	end

end

In [ ]:
τ_f = 0.0001
τ_s = 0.02
τ_us = 0.4

Ithr_f = 150e-9
Igain_f = 700e-9
Ilin_f = 250e-9

Ithr_s = 110e-9
Igain_s = 250e-9
Ilin_s = 50e-9

In [ ]:
IV_Fast(I_eq) = I_eq - Imode_sigmoid_eval(I_eq, (Ithr=Ithr_f, Igain=Igain_f, Ilin=Ilin_f))
IV_Slow(I_eq) = 2*I_eq - Imode_sigmoid_eval(I_eq, (Ithr=Ithr_f, Igain=Igain_f, Ilin=Ilin_f))

plot([IV_Fast, IV_Slow], 0, 1e-6, legend = false, xlabel="Ieq (A)", ylabel="Iapp (A)", layout = (1, 2), size = (800, 400))

In [ ]:
IV_SlowB(I_eq) = 2*I_eq - Imode_sigmoid_eval(I_eq, (Ithr=Ithr_f, Igain=Igain_f, Ilin=Ilin_f)) - Imode_sigmoid_eval(I_eq, (Ithr=Ithr_s, Igain=Igain_s, Ilin=Ilin_s))
IV_UslowB(I_eq) = 3*I_eq - Imode_sigmoid_eval(I_eq, (Ithr=Ithr_f, Igain=Igain_f, Ilin=Ilin_f)) - Imode_sigmoid_eval(I_eq, (Ithr=Ithr_s, Igain=Igain_s, Ilin=Ilin_s))

plot([IV_Fast, IV_SlowB, IV_UslowB], 0, 8e-7, legend = false, xlabel="Ieq (A)", ylabel="Iapp (A)", minorticks=true, layout = (1, 3), size = (1200, 400))

In [ ]:
Tfinal = 2.
tspan = (0.0, Tfinal)

Iapp = 5e-7

x0 = [100e-9, 100e-9]

pars = (Iapp, Ithr_f, Igain_f, Ilin_f)

prob = ODEProblem(SpikingModel!, x0, tspan, pars)
sol = solve(prob, Rodas4P(), abstol=1e-15, reltol=1e-12)
plot(sol, xlabel="Time (s)", ylabel="Current (A)", label=[L"I_f" L"I_s"], size = (800, 400))

In [ ]:
Tfinal = 3.
tspan = (0.0, Tfinal)

Iapp = 390e-9

x0 = [1e-9, 1e-9, 1e-9]

pars = (Iapp, Ithr_f, Igain_f, Ilin_f, Ithr_s, Igain_s, Ilin_s)

prob = ODEProblem(BurstingModel_inact!, x0, tspan, pars)
sol = solve(prob, Rodas4P(), abstol=1e-15, reltol=1e-12)
plot(sol, idxs=[1], xlabel="Time (s)", ylabel="Current (A)", label=[L"I_f" L"I_s" L"I_us"])